# 🦈 Matrix Compression with SVD

🎯 **Goal:** make a photo smaller, but keep it looking good.

📷 Photo ➡️ 🔢 Numbers ➡️ 🍰 Layers ➡️ ✂️ Keep k layers ➡️ 🖼️ Smaller photo

## 📦 Libraries and Tools
Load the tools we need.

In [ ]:
# 🔢 Numbers and tables (matrices)
import numpy as np

# 🖼️ Open photos
from PIL import Image

# 📊 Draw pictures and charts
import matplotlib.pyplot as plt

# 📁 Where to save the figures for the slides
from pathlib import Path
OUT_DIR = Path("image")
OUT_DIR.mkdir(exist_ok=True)

print("✅ Tools loaded successfully!")

## 🔢 Step 1 — A photo is numbers
A photo is made of tiny dots called **pixels**.

Each colour pixel has **3 numbers**: 🔴 **R**ed, 🟢 **G**reen, 🔵 **B**lue, each from **0 (none)** to **255 (full)**.

Mixing the 3 numbers gives the colour of that pixel.

A table of numbers is called a **matrix**.

In [ ]:
# 🦈 Open the photo as numbers
img = np.asarray(Image.open("images/shark.jpg").convert("RGB"), dtype=float)

# 🔍 Zoom into 1 pixel
row, col = 400, 500
r, g, b = img[row, col].astype(int)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={"width_ratios": [1.5, 1]})
ax1.imshow(img.astype(np.uint8))
ax1.add_patch(plt.Circle((col, row), 20, fill=False, color="red", lw=3))
ax1.annotate("zoom", (col + 15, row - 15), xytext=(col + 140, row - 140),
             color="red", fontsize=15, arrowprops={"color": "red", "lw": 2})
ax1.set_title("The photo", fontsize=16)
ax1.axis("off")

# 🟥🟩🟦 1 pixel, with its 3 numbers inside
ax2.imshow(img[row:row + 1, col:col + 1].astype(np.uint8))
ax2.text(0, 0, f"R = {r}\nG = {g}\nB = {b}", ha="center", va="center", fontsize=28,
         color="white" if (r + g + b) / 3 < 128 else "black")
ax2.set_title("Zoomed in: 1 pixel", fontsize=16)
ax2.axis("off")

fig.savefig(OUT_DIR / "00_photo_is_numbers.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Pixel at row {row}, column {col}: R = {r}, G = {g}, B = {b}")
print("✅ m x n =", img.shape[0], "x", img.shape[1], "pixels, 3 colours (R, G, B)")

## 🍰 Step 2 — Split the photo into layers
**SVD** splits the photo into layers.

Layer 1 is the most important ⭐, then layer 2, and so on.

The more layers we add, the clearer the photo gets. (We do this for each colour 🔴🟢🔵, then put them back together.)

In [ ]:
# 🍰 SVD on each colour (R, G, B): U, S (importance of each layer), Vt
svds = [np.linalg.svd(img[:, :, c], full_matrices=False) for c in range(3)]

fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
for ax, k in zip(axes, [1, 3, 10, 50]):
    # ➕ add up the first k layers of each colour, then put R, G, B back together
    A_k = np.stack([U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :] for U, S, Vt in svds], axis=2)
    ax.imshow(np.clip(A_k, 0, 255).astype(np.uint8))
    ax.set_title(f"k = {k}", fontsize=15)
    ax.axis("off")

fig.savefig(OUT_DIR / "01_svd_layers.png", dpi=150, bbox_inches="tight")
plt.show()
for name, (U, S, Vt) in zip("RGB", svds):
    print(f"✅ {name}: importance of layers 1, 2, 3:", S[:3].round(0), "... layer 200:", S[199].round(0))

## ✂️ Step 3 — Keep only k layers
**k** = how many layers we keep. We throw away the rest.

- 🔻 Small k → small file, blurry photo
- 🔺 Big k → bigger file, clear photo

We check two things:
- 💾 **Storage:** how much space we use (the original = 100%)
- 🎯 **Quality:** how close it looks to the original, in **dB** (30 dB or more = looks very good)

In [ ]:
# 🍰 Split each colour (R, G, B) into layers
def load(path):
    img = np.asarray(Image.open(path).convert("RGB"), dtype=float)
    return img, [np.linalg.svd(img[:, :, c], full_matrices=False) for c in range(3)]


# ✂️ Rebuild the photo from the first k layers
def compress(svds, k):
    channels = [U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :] for U, S, Vt in svds]
    return np.clip(np.stack(channels, axis=2), 0, 255).astype(np.uint8)


# 💾 Space used compared with the original
def storage(img, k):
    m, n, _ = img.shape
    return k * (m + n + 1) / (m * n)


# 🎯 Quality score (dB)
def psnr(img, approx):
    return 10 * np.log10(255 ** 2 / np.mean((img - approx) ** 2))


data = {name: load(f"images/{name.lower()}.jpg") for name in ["Shark", "Deer", "Leopard"]}
print("✅ Ready!")

## 🖼️ Step 4 — Results
Each picture shows: **k** · 💾 **storage** · 🎯 **quality**.

👀 Look how the photo gets clearer as k gets bigger.

In [ ]:
for name, (img, svds) in data.items():
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
    axes[0].imshow(img.astype(np.uint8))
    axes[0].set_title("Original · 100%", fontsize=14)

    # ✂️ More layers = sharper photo, bigger file
    for ax, k in zip(axes[1:], [5, 20, 200]):
        approx = compress(svds, k)
        ax.imshow(approx)
        ax.set_title(f"k = {k} · {storage(img, k):.0%} · {psnr(img, approx):.0f} dB", fontsize=14)

    for ax in axes:
        ax.axis("off")
    fig.savefig(OUT_DIR / f"02_results_{name.lower()}.png", dpi=150, bbox_inches="tight")
    plt.show()

## 📈 Step 5 — How many layers do we need?
- ⬆️ Higher line = clearer photo
- ⬅️ More to the left = smaller file
- ➖ Dashed line = 30 dB (looks very good 😍)

Where a line crosses the dashed line, the photo looks good.
🦌 The deer needs about **25%** of the space. 🦈 The shark and 🐆 leopard need about **half**.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
for (name, (img, svds)), color in zip(data.items(), ["#2a78d6", "#eb6834", "#1baf7a"]):
    ks = list(range(1, 20)) + list(range(20, 401, 10))
    ax.plot([storage(img, k) * 100 for k in ks], [psnr(img, compress(svds, k)) for k in ks],
            color=color, lw=2.5, label=name)

# 😍 30 dB = looks very good
ax.axhline(30, color="gray", ls="--")
ax.text(1, 30.5, "30 dB = looks very good", color="gray")

ax.set(xlim=(0, 100), xlabel="storage used (%)", ylabel="quality (dB)",
       title="Quality vs. storage")
ax.legend(loc="lower right")
fig.savefig(OUT_DIR / "03_quality_vs_storage.png", dpi=150, bbox_inches="tight")
plt.show()

## 🏁 Summary

- 🔢 A photo is a table of numbers (a matrix).

- 🍰 SVD splits it into layers, most important first.

- ✂️ We keep only k layers: small k = small file but blurry, big k = clear but bigger file.

- 🦈 With 5% of the space we can already see the shark. With about half, it looks very good.